# Function Testing Notebook

Author: Pete King

This notebook tests custom functions developed in the various helper modules to verify proper operation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

from helpers import data_prep as dp

DATA_DIR = Path.cwd().parent / 'data'
DATA_FILE = DATA_DIR / 'etf_raw_data.csv'
ETF='SPY'

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Import and inspect ETF price data

In [2]:
df = pd.read_csv(
    DATA_FILE,
    index_col='date',
    parse_dates=True
)
df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.175390,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.347324,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.398920,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.656818,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.760015,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-09,91.459999,73.750000,437.910004,80.279999,95.430000,261.959991,109.489998,610.190002,679.909973,110.959999,...,51.669998,57.330002,51.330002,172.190002,142.070007,83.449997,42.73,47.150002,149.330002,112.739998
2026-04-10,91.489998,73.639999,437.130005,79.959999,95.269997,261.299988,109.199997,611.070007,679.460022,111.019997,...,51.959999,56.939999,50.770000,171.520004,142.619995,82.370003,42.82,46.959999,147.309998,112.889999
2026-04-13,91.489998,73.800003,435.359985,80.260002,95.480003,265.070007,109.620003,617.390015,686.099976,111.279999,...,52.189999,57.110001,51.660000,172.729996,145.610001,81.550003,43.02,46.389999,147.970001,113.919998


## Compute and display daily return

Here we test the ability of the log_return function to compute daily returns and inspect the results.

In [3]:
test_df = df[[ETF]]
etf_return = test_df['SPY'].rolling(2).apply(dp.log_return, raw=True)
test_df[ETF + '_return'] = etf_return.values
test_df

,SPY,SPY_return
date,,
1993-01-29,24.175390,NaN
1993-02-01,24.347324,0.007087
1993-02-02,24.398920,0.002117
1993-02-03,24.656818,0.010515
1993-02-04,24.760015,0.004177
...,...,...
2026-04-09,679.909973,0.005753
2026-04-10,679.460022,-0.000662
2026-04-13,686.099976,0.009725


In [4]:
chart = alt.Chart(test_df.dropna().reset_index()).mark_circle(size=10).encode(
    x='date:T',
    y=ETF + '_return:Q'
)
chart.properties(
    title='S&P 500 ETF (SPY) - Historical Return',
    height=200, width=800
)

alt.Chart(...)

## Discussion

From the chart we can see that the mean of daily returns appears to be nearly zero -- an empirical justification of the zero mean assumption for expected return (E\[R\]).

Since volatility for an asset is a measure of the deviation of returns from expected return, we can get a feel for an asset's volatility just by inspecting the plot.  We see a general trend of baseline low volatility (for example, from Jan 2004 to Jan 2007), with periods of high volatility that tend to gradually revert to baseline (for example during the 'Great Recession', from late 2007, spiking in late 2008 / early 2009, and gradually reverting to a lower baseline by roughly 2012).

In [5]:
etf_return.describe()

count    8358.000000
mean        0.000403
std         0.011724
min        -0.115887
25%        -0.004331
50%         0.000681
75%         0.005924
max         0.135578
Name: SPY, dtype: float64

The main idea with this project is to think of the daily return for each asset as a random variable (R), and then investigate its statistical properties.  

***Right away, from this simple statistical description (above), we can get an idea of what to expect for the properties of an asset's return (R):***
 - Estimated **expected return** (E\[R\]): 0.04 percent (very close to zero)
 - Estimated long-term (baseline) **volatility**: 1.17 percent

*Note that the financial term "volatility" can have mean interpretations, but here we mean ***realized volatility***, also known as "historical" volatility, or the standard deviation of return (R), assuming zero mean.*

In [6]:
# Compute using the zero-mean assumption for expected return
vol = np.sqrt(
    np.sum(etf_return.dropna().values**2) / len(etf_return.dropna())
)
print(f'Estimated long-term volatility with zero-mean assumption: \
        {vol * 100:2.2f} percent'
     )

Estimated long-term volatility with zero-mean assumption:         1.17 percent
